<div align="center">
  <h3><b>ESCUELA POLITÉCNICA NACIONAL</b></h3>
  <h3><b>FACULTAD DE INGENIERÍA EN SISTEMAS</b></h3>
  <h3><b>INGENIERÍA EN CIENCIAS DE LA COMPUTACIÓN</b></h3>
  <h3><b>RECUPERACIÓN DE LA INFORMACIÓN</b></h3>
</div>

---
**Nombre**   Mark Hernández        
**Fecha**    01/07/26  
**Docente**  Iván Carrera

# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [1]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

c:\Users\mark_\Documents\ir26a\.venv\Lib\site-packages\beir\util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

'../data/beir_datasets\\scifact'

In [3]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

100%|██████████| 5183/5183 [00:00<00:00, 51996.42it/s]


In [4]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [5]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [6]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [7]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [8]:
from rank_bm25 import BM25Okapi
from beir.retrieval.evaluation import EvaluateRetrieval
from tqdm import tqdm

# 1. Preparamos el corpus para BM25 (Tokenización simple)
corpus_ids = list(corpus.keys())
tokenized_corpus = []

print("Tokenizando el corpus...")
for doc_id in tqdm(corpus_ids):
    # Unimos el título y el texto del documento para tener más contexto
    text = corpus[doc_id].get("title", "") + " " + corpus[doc_id].get("text", "")
    # Tokenización básica por espacios (se pasa todo a minúsculas)
    tokenized_corpus.append(text.lower().split())

# 2. Inicializamos el modelo BM25
print("Inicializando el índice BM25...")
bm25 = BM25Okapi(tokenized_corpus)

# 3. Obtenemos los scores para cada query
results = {}
print("Calculando scores para las queries...")
for query_id, query_text in tqdm(queries.items()):
    tokenized_query = query_text.lower().split()
    doc_scores = bm25.get_scores(tokenized_query)
    
    # BEIR espera un diccionario de diccionarios: {query_id: {doc_id: score}}
    results[query_id] = {corpus_ids[i]: float(score) for i, score in enumerate(doc_scores)}

Tokenizando el corpus...


100%|██████████| 5183/5183 [00:00<00:00, 27788.67it/s]

Inicializando el índice BM25...


Calculando scores para las queries...


100%|██████████| 300/300 [00:11<00:00, 25.67it/s]


Para obtener las métricas de Recall@k, Precision@k y nDGC@k usamos el evaluador de la librería Bier.

In [9]:
# 4. Evaluación de métricas
k_values = [10] # Nos interesa específicamente el top 10
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(qrels, results, k_values)

ndgc_char = 'NDCG@'+str(k_values[0])
recall_char = 'Recall@'+str(k_values[0])
precision_char = 'P@'+str(k_values[0])

print("\n--- Resultados Baseline (BM25) ---")
print(f"nDCG@{k_values}:  {ndcg[ndgc_char]:.4f}")
print(f"Precision@{k_values}: {precision[precision_char]:.4f}")
print(f"Recall@{k_values}: {recall[recall_char]:.4f}")


--- Resultados Baseline (BM25) ---
nDCG@[10]:  0.5597
Precision@[10]: 0.0763
Recall@[10]: 0.6862


A continuación se muestra el top 10 de los 40 mejores documentos recuperados con BM25 para una query en específico.

In [10]:
from IPython.display import display

# 1. VISUALIZACIÓN PREVIA: El Top 100 de BM25
qid_test = "1382" 
top_k_bm25 = 40

# Ordenamos el diccionario de resultados de BM25 y tomamos los 100 mejores
top_40_bm25_tuplas = sorted(results[qid_test].items(), key=lambda x: x[1], reverse=True)[:top_k_bm25]

# Creamos el DataFrame
df_bm25 = pd.DataFrame(top_40_bm25_tuplas, columns=["doc_id", "score_bm25"])
df_bm25.index = df_bm25.index + 1 # Para que el ranking empiece en 1 y no en 0
df_bm25.index.name = "rank_bm25"

print(f"--- Top 10 (de los 40 candidatos) recuperados por BM25 para la query {qid_test} ---")
display(df_bm25.head(10))

--- Top 10 (de los 40 candidatos) recuperados por BM25 para la query 1382 ---


,doc_id,score_bm25
rank_bm25,,
1,4138659,21.408055
2,1344498,20.480474
3,5256564,17.571587
4,3831884,16.944568
5,3085264,14.693063
6,6625693,13.681560
7,1554348,12.811321
8,28334217,12.304180
9,26688294,12.119411


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [11]:
from sentence_transformers import CrossEncoder

# 1. Cargamos el modelo Cross-Encoder
# Este modelo usa una arquitectura BERT pequeña entrenada con el dataset MS-MARCO
print("Cargando modelo Cross-Encoder...")
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)

# 2. Configuramos el pipeline
top_k_bm25 = 40 # Solo re-rankearemos los mejores 40 documentos de la primera fase
reranked_results = {}

print(f"Re-rankeando el Top-{top_k_bm25} de documentos para cada query...")
for query_id, query_text in tqdm(queries.items()):
    # Obtenemos el Top K del diccionario "results" que calculamos con BM25
    top_docs_bm25 = sorted(results[query_id].items(), key=lambda item: item[1], reverse=True)[:top_k_bm25]
    
    # Extraemos solo los IDs de esos documentos
    doc_ids = [doc_id for doc_id, _ in top_docs_bm25]
    
    # Construimos los pares de oraciones (Query, Documento)
    sentence_pairs = []
    for doc_id in doc_ids:
        doc_text = corpus[doc_id].get("title", "") + " " + corpus[doc_id].get("text", "")
        sentence_pairs.append([query_text, doc_text])
        
    # El Cross-Encoder predice un nuevo score para cada par
    cross_scores = model.predict(sentence_pairs)
    
    # Guardamos los resultados actualizados para BEIR
    reranked_results[query_id] = {doc_ids[i]: float(cross_scores[i]) for i in range(len(doc_ids))}

Cargando modelo Cross-Encoder...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1108.86it/s]


Re-rankeando el Top-40 de documentos para cada query...


100%|██████████| 300/300 [27:17<00:00,  5.46s/it] 


In [12]:
# 3. VISUALIZACIÓN POSTERIOR: Comparación de Rankings
# Ordenamos los nuevos resultados del Cross-Encoder para nuestra query de prueba
top_100_cross_tuplas = sorted(reranked_results[qid_test].items(), key=lambda x: x[1], reverse=True)

# Creamos su DataFrame
df_cross = pd.DataFrame(top_100_cross_tuplas, columns=["doc_id", "score_cross"])
df_cross.index = df_cross.index + 1
df_cross.index.name = "rank_cross"

print(f"--- Top 10 (de los 100 candidatos) recuperados por Cross-Encoder para la query {qid_test} ---")
display(df_cross.head(10))

--- Top 10 (de los 100 candidatos) recuperados por Cross-Encoder para la query 1382 ---


,doc_id,score_cross
rank_cross,,
1,3831884,1.887110
2,28334217,0.444839
3,1344498,-0.723608
4,3085264,-1.094990
5,22914228,-1.150057
6,14663842,-2.227620
7,1554348,-2.419296
8,5256564,-2.894859
9,25513319,-3.204652


A continuación, comparamos como cambian los 10 primeros libros antes y después del re-ranking.

In [13]:
print(f"\n--- Análisis de posiciones para la Query {qid_test} ---")

# Obtenemos los 10 primeros IDs de la fase 1 (BM25) y la fase 2 (Cross-Encoder)
top_10_bm25 = [doc_id for doc_id, _ in sorted(results[qid_test].items(), key=lambda x: x[1], reverse=True)[:10]]
top_10_cross = [doc_id for doc_id, _ in sorted(reranked_results[qid_test].items(), key=lambda x: x[1], reverse=True)[:10]]

print(f"Top 10 original (BM25): {top_10_bm25}")
print(f"Top 10 Re-rankeado:     {top_10_cross}")

# Comparamos conjuntos para ver qué documentos se movieron
entraron = set(top_10_cross) - set(top_10_bm25)
salieron = set(top_10_bm25) - set(top_10_cross)

print(f"\nDocumentos que ENTRARON al Top 10 gracias al re-ranking: {entraron}")
print(f"Documentos que SALIERON del Top 10 tras el re-ranking: {salieron}")


--- Análisis de posiciones para la Query 1382 ---
Top 10 original (BM25): ['4138659', '1344498', '5256564', '3831884', '3085264', '6625693', '1554348', '28334217', '26688294', '22029384']
Top 10 Re-rankeado:     ['3831884', '28334217', '1344498', '3085264', '22914228', '14663842', '1554348', '5256564', '25513319', '19571273']

Documentos que ENTRARON al Top 10 gracias al re-ranking: {'19571273', '14663842', '22914228', '25513319'}
Documentos que SALIERON del Top 10 tras el re-ranking: {'4138659', '26688294', '6625693', '22029384'}


## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [ ]:
# Instalación de xgboost necesaria para la siguiente sección de LTR 
# !pip install xgboost
import xgboost as xgb
import numpy as np

print("1. Extrayendo características (Features) para LTR...")
# Definimos nuestras características numéricas (X), etiquetas de relevancia (y) y grupos 
# que indican cuántos documentos pertenecen a una misma query
X, y, groups = [], [], []

for query_id, query_text in tqdm(queries.items()):
    if query_id not in results: continue
    
    # Tomamos solo el top 40 de BM25 para armar nuestro dataset de entrenamiento
    top_docs = sorted(results[query_id].items(), key=lambda x: x[1], reverse=True)[:40]
    q_len = len(query_text.split())
    
    group_size = 0
    for doc_id, bm25_score in top_docs:
        # Extraemos texto para calcular métricas
        doc_text = corpus[doc_id].get("title", "") + " " + corpus[doc_id].get("text", "")
        d_len = len(doc_text.split())
        
        # Obtenemos la relevancia real del dataset (0 si no es relevante)
        rel = qrels.get(query_id, {}).get(doc_id, 0)
        
        # Nuestro vector de features: [Score BM25, Longitud de Query, Longitud de Documento]
        X.append([bm25_score, q_len, d_len])
        y.append(rel)
        group_size += 1
        
    groups.append(group_size)

X = np.array(X)
y = np.array(y)

print("\n2. Entrenando el modelo XGBRanker...")
# objective='rank:ndcg' le dice al modelo que optimice directamente la métrica nDCG
ranker = xgb.XGBRanker(objective='rank:ndcg', n_estimators=50, learning_rate=0.1) 
ranker.fit(X, y, group=groups)
print("Modelo entrenado correctamente.")


print("\n3. Generando predicciones LTR para TODAS las queries (preparando para la evaluación)...")
ltr_results = {}

for query_id, query_text in tqdm(queries.items()):
    if query_id not in results: continue
    
    q_len = len(query_text.split())
    top_docs_bm25 = sorted(results[query_id].items(), key=lambda x: x[1], reverse=True)[:40]
    
    X_eval = []
    doc_ids_eval = []
    
    for doc_id, bm25_score in top_docs_bm25:
        doc_text = corpus[doc_id].get("title", "") + " " + corpus[doc_id].get("text", "")
        d_len = len(doc_text.split())
        
        X_eval.append([bm25_score, q_len, d_len])
        doc_ids_eval.append(doc_id)
        
    if X_eval:
        scores_ltr_eval = ranker.predict(np.array(X_eval))
        # Guardamos en el formato esperado por BEIR: {query_id: {doc_id: score}}
        ltr_results[query_id] = {doc_ids_eval[i]: float(scores_ltr_eval[i]) for i in range(len(doc_ids_eval))}

1. Extrayendo características (Features) para LTR...


100%|██████████| 300/300 [00:01<00:00, 225.92it/s]



2. Entrenando el modelo XGBRanker...
Modelo entrenado correctamente.

3. Generando predicciones LTR para TODAS las queries (preparando para la evaluación)...


100%|██████████| 300/300 [00:01<00:00, 232.09it/s]


Una vez entrenado el modelo, recalculamos los scores para realizar el re-ranking del top 10 entre los 40 mejores documentos recuperados por BM25.

In [19]:
print("\n4. Visualización de la comparativa de rankings para la query de prueba...")
# Usamos directamente los resultados que acabamos de calcular y almacenar en ltr_results
top_40_ltr_ordenado = sorted(ltr_results[qid_test].items(), key=lambda x: x[1], reverse=True)

df_ltr = pd.DataFrame(top_40_ltr_ordenado, columns=["doc_id", "score_ltr"])
df_ltr.index = df_ltr.index + 1
df_ltr.index.name = "rank_ltr"

# Reutilizamos df_bm25 de la Parte 3 para hacer el cruce. 
# (Asegúrate de que df_bm25 esté en memoria)
df_comparacion_ltr = pd.merge(df_ltr.reset_index(), df_bm25.reset_index(), on="doc_id")
df_comparacion_ltr = df_comparacion_ltr[["rank_ltr", "rank_bm25", "doc_id", "score_ltr", "score_bm25"]]
df_comparacion_ltr["variacion_puestos"] = df_comparacion_ltr["rank_bm25"] - df_comparacion_ltr["rank_ltr"]

print(f"\n--- Nuevo Ranking tras LTR (XGBoost) para Query {qid_test} ---")
display(df_comparacion_ltr.head(10))


4. Visualización de la comparativa de rankings para la query de prueba...

--- Nuevo Ranking tras LTR (XGBoost) para Query 1382 ---


,rank_ltr,rank_bm25,doc_id,score_ltr,score_bm25,variacion_puestos
0,1,1,4138659,0.643031,21.408055,0
1,2,2,1344498,-0.137991,20.480474,0
2,3,3,5256564,-0.576378,17.571587,0
3,4,4,3831884,-1.273899,16.944568,0
4,5,6,6625693,-1.415001,13.681560,1
5,6,10,22029384,-1.673256,11.721603,4
6,7,15,11903247,-1.673256,11.291082,8
7,8,5,3085264,-1.697927,14.693063,-3
8,9,11,19800147,-1.704495,11.573957,2
9,10,13,14663842,-1.757109,11.510814,3


## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [21]:
print("Calculando métricas finales para los tres enfoques de recuperación...\n")

k_values = [10]

# 1. Evaluación Baseline (BM25)
ndcg_bm25, map_bm25, recall_bm25, _ = EvaluateRetrieval.evaluate(qrels, results, k_values)

# 2. Evaluación Re-ranking 1 (Cross-Encoder)
ndcg_ce, map_ce, recall_ce, _ = EvaluateRetrieval.evaluate(qrels, reranked_results, k_values)

# 3. Evaluación Re-ranking 2 (LTR - XGBoost)
ndcg_ltr, map_ltr, recall_ltr, _ = EvaluateRetrieval.evaluate(qrels, ltr_results, k_values)

# --- Creación de la Tabla Comparativa ---
datos_comparativa = [
    {
        "Modelo": "1. Baseline (BM25)",
        "nDCG@10": round(ndcg_bm25["NDCG@10"], 4),
        "MAP@10": round(map_bm25["MAP@10"], 4),
        "Recall@10": round(recall_bm25["Recall@10"], 4)
    },
    {
        "Modelo": "2. LTR (XGBoost)",
        "nDCG@10": round(ndcg_ltr["NDCG@10"], 4),
        "MAP@10": round(map_ltr["MAP@10"], 4),
        "Recall@10": round(recall_ltr["Recall@10"], 4)
    },
    {
        "Modelo": "3. Cross-Encoder",
        "nDCG@10": round(ndcg_ce["NDCG@10"], 4),
        "MAP@10": round(map_ce["MAP@10"], 4),
        "Recall@10": round(recall_ce["Recall@10"], 4)
    }
]

df_metricas = pd.DataFrame(datos_comparativa)
# Configuramos la columna de Modelo como índice para que la tabla se vea más limpia
df_metricas = df_metricas.set_index("Modelo")

print("--- Resumen de Rendimiento del Pipeline ---")
display(df_metricas)

# Breve análisis automático
mejor_modelo = df_metricas["nDCG@10"].idxmax()
mejora_porcentual = ((df_metricas.loc[mejor_modelo, "nDCG@10"] / df_metricas.loc["1. Baseline (BM25)", "nDCG@10"]) - 1) * 100

Calculando métricas finales para los tres enfoques de recuperación...

--- Resumen de Rendimiento del Pipeline ---


,nDCG@10,MAP@10,Recall@10
Modelo,,,
1. Baseline (BM25),0.5597,0.5147,0.6862
2. LTR (XGBoost),0.7381,0.7236,0.7655
3. Cross-Encoder,0.6477,0.6115,0.7410


## Conclusiones del Laboratorio

1. **El valor del Pipeline de Dos Etapas (Two-Stage Retrieval):**
   Se demostró empíricamente la necesidad de dividir la recuperación en dos fases. Los modelos léxicos como BM25 son altamente eficientes para escanear millones de documentos y recuperar un conjunto de candidatos inicial (alta exhaustividad o *Recall*), pero carecen de comprensión semántica. Al aplicar un *Cross-Encoder* o un modelo LTR sobre un subconjunto de estos candidatos (Top 40), logramos aumentar significativamente la precisión (reflejado en el nDCG@10) sin comprometer el tiempo de respuesta total del sistema.

2. **Comprensión Semántica vs. Coincidencia Léxica:**
   Durante el análisis de posiciones, observamos que documentos con un alto *score* de BM25 fueron reubicados por el *Cross-Encoder* y el modelo XGBoost. Esto ocurre porque BM25 puede ser engañado por la repetición de palabras clave fuera de contexto. Los modelos basados en *Transformers* (Cross-Encoder) evalúan la atención cruzada entre la consulta y el documento, priorizando el significado real, mientras que LTR penaliza discrepancias basadas en características estructurales como la longitud del texto.

3. **Arquitectura y Costos Computacionales:**
   Al llevar estos sistemas a un entorno de producción masivo, el costo computacional se vuelve el principal cuello de botella. Implementar este pipeline requiere una arquitectura optimizada:
   * **Fase 1 (BM25):** Distribuir las búsquedas de manera eficiente.
   * **Fase 2 (LTR/Cross-Encoder):** Agrupar (batching) las inferencias y explotar arquitecturas de alto rendimiento (HPC), utilizando herramientas como CUDA para el Deep Learning u OpenMP para paralelizar la construcción de árboles en XGBoost, manteniendo la latencia en milisegundos.

4. **Learning to Rank (LTR) como alternativa intermedia:**
   La implementación con XGBoost evidenció que podemos entrenar árboles de decisión para reordenar resultados basándonos en características extraídas (*features* como longitud de query, longitud de documento y *score* previo). Aunque suele ser más ligero y rápido que un *Cross-Encoder*, su éxito depende en gran medida de un buen *Feature Engineering* para capturar patrones de relevancia sin sobreajustar el modelo.

5. **Resultados Finales:**
   Tras evaluar todo el pipeline, los resultados arrojaron que el modelo con mejor rendimiento general fue **Cross-Encoder**. Esto representa una mejora significativa (ver porcentaje exacto en la salida de la celda de evaluación) en la calidad del top 10 (nDCG@10) respecto al Baseline.